In [ ]:
import os, sys
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
# Cria a conexão Spark

# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")

In [ ]:
project_path    = os.getcwd()
data_path_root  = "C:\\Marco Conti\\Projetos\\Dados\\"

Seleciona os dados de temperatura

In [ ]:
df_tempetatura_2023 = spark.read.parquet(f"{data_path_root}ERA5-temperaturas\\2023\\ERA5_temperatura.parquet")
print(f"df_tempetatura_2023 tem: {df_tempetatura_2023.count()} registros")
df_tempetatura_2024 = spark.read.parquet(f"{data_path_root}ERA5-temperaturas\\2024\\ERA5_temperatura.parquet")
print(f"df_tempetatura_2024 tem: {df_tempetatura_2024.count()} registros")
df_tempetatura_2025 = spark.read.parquet(f"{data_path_root}ERA5-temperaturas\\2025\\ERA5_temperatura.parquet")
print(f"df_tempetatura_2025 tem: {df_tempetatura_2024.count()} registros")

df_tempetatura = df_tempetatura_2023.union(df_tempetatura_2024).union(df_tempetatura_2025)

df_tempetatura.printSchema()
df_tempetatura.limit(10).show()

In [ ]:
# df_tempetatura_2023.filter("latitude = -34.0 and longitude = -67.0").orderBy("valor").show(10,False)

Obtem as estatísticas de temperatura por Ano:
- Temperatura mínima
- Temperatura máxima
- Temperatura média
- Percentil 5%
- Percentil 90%

In [ ]:
# Criar as colunas min, max, med e percentis por ANO*
# * De acordo com documento elaborado por Sara Lopes de Moraes para o VERACIS

df_base = (
    df_tempetatura
        .withColumn("ano", F.year("data_medicao"))
        .withColumn("mes", F.month("data_medicao"))
)

df_base.printSchema()
df_base.show(10, truncate=False)

df_stats = (
    df_base
    .groupBy("ano"
           #,"mes"
            ,"latitude"
            ,"longitude"
    )
    .agg(F.min("valor").alias("temp_min_ano")
        ,F.max("valor").alias("temp_max_ano")
        ,F.avg("valor").alias("temp_media_ano")
        ,F.expr("percentile_approx(valor, 0.05)").alias("percentil_05_ano")
        ,F.expr("percentile_approx(valor, 0.95)").alias("percentil_95_ano")
    )
)

print("df_stats.count()", df_stats.count())
df_stats.printSchema()
df_stats.show(10, truncate=False)

Anexar dados estatíscos e percentis ao dado diário

Será utilizados para calcular as ondas de calor (periodos de dias consecutivos)

In [ ]:
df_dia = (
    df_base.alias("base")
    .join(
        df_stats.alias("stats"),
        [
            F.col("base.ano") == F.col("stats.ano"),
            F.col("base.latitude") == F.col("stats.latitude"),
            F.col("base.longitude") == F.col("stats.longitude"),
        ],
        how="left",
    )
    .select(
        "stats.ano",
        "base.mes",
        "stats.latitude",
        "stats.longitude",
        "stats.temp_min_ano",
        "stats.temp_max_ano",
        "stats.temp_media_ano",
        "stats.percentil_05_ano",
        "stats.percentil_95_ano",
        "base.data_medicao",
        "base.indicador",
        "base.valor",
        "base.unidade_medida"
    )
)

# print("Número de registros no DataFrame diário:", df_dia.count()) # 151662
# df_dia.printSchema()
df_dia.filter("ano = 2023 and latitude = -23 and longitude = -46").orderBy("valor").show(10, truncate=False)

In [ ]:
# Número de registros no DataFrame diário: 27.703.592

Classifica os valores extremos para calor e frio
- Calor: temperatura diaria é maior ou igual ao percentil 95
- Frio: temperatura diária é menor ou igual ao percentil 5

In [ ]:
df_dia = (
    df_dia
    .withColumn(
        "extremo_alto",
        F.when(F.col("valor") >= F.col("percentil_95_ano")
              ,(F.col("valor") - F.col("percentil_95_ano"))).otherwise(0))
    .withColumn(
        "extremo_baixo",
        F.when(F.col("valor") <= F.col("percentil_05_ano")
              ,(F.col("percentil_05_ano") - F.col("valor"))).otherwise(0)
    )
)

print(f"registros no df_dia: ", df_dia.count())
df_dia.printSchema()
df_dia.filter("extremo_baixo != 0 or extremo_alto != 0").show(10, truncate=False)
# (df_dia
#     .select('data_medicao', 'latitude', 'longitude', 'percentil_05_ano', 'valor', 'percentil_95_ano'
#            ,'extremo_alto', 'extremo_baixo' )
#     .orderBy("latitude", "longitude", "data_medicao")
#     .show(1000, truncate=False))

Separa somente os dias quentes ou frios de acordo com a regra definida no passo anterior

In [ ]:

df_flag = \
    (df_dia.withColumn("flag_dia_quente"
                     ,F.when(F.col("valor") > F.col("percentil_95_ano"), 1).otherwise(0))
           .withColumn("flag_dia_frio"
                      ,F.when(F.col("valor") < F.col("percentil_05_ano"), 1).otherwise(0)))

# Mantém apenas os dias que atenderam ao critério de dias quentes ou frios
df_quentes = df_flag.filter(F.col("flag_dia_quente") == 1)
df_frios   = df_flag.filter(F.col("flag_dia_frio") == 1)

# Janela ordenada por data para cada ponto geográfico, assim será possível identificar os períodos consecutivos de ondas de calor ou frio
janela_loc = Window.partitionBy("latitude", "longitude").orderBy("data_medicao")

# Ao subtrair a ordem do registro (rn) da data, dias consecutivos geram
# exatamente o mesmo identificador de grupo (grupo_id)
df_eventos = \
    (df_quentes
        .withColumn("rn", F.row_number().over(janela_loc)) \
        .withColumn("grupo_id", F.expr("date_sub(data_medicao, CAST(rn AS INT))")))

df_eventos_frio = \
    (df_frios
        .withColumn("rn", F.row_number().over(janela_loc))
        .withColumn("grupo_id", F.expr("date_sub(data_medicao, CAST(rn AS INT))"))
)


In [ ]:
# 1. Identificação dos Eventos de CALOR e Validação da Duração (Global, sem quebra de mês/ano, usando apenas as coordenadas geográfica)
df_eventos_duracao = (
    df_eventos
    .groupBy("latitude", "longitude", "grupo_id")
    .agg(
        F.count("data_medicao").alias("duracao_total_onda"),
        F.min("data_medicao").alias("inicio_onda"),
        F.max("data_medicao").alias("fim_onda")
    )
    .filter(F.col("duracao_total_onda") > 2) # Filtra apenas eventos reais (> 2 dias)
)

# 2. Retornar os DIAS INDIVIDUAIS das ondas válidas mantendo os metadados da onda
df_dias_em_onda = (
    df_eventos
    .join(df_eventos_duracao, ["latitude", "longitude", "grupo_id"], "inner")
    .select("latitude", 
            "longitude", 
            "data_medicao", 
            "ano", 
            "mes", 
            "grupo_id", 
            "duracao_total_onda",
            "valor"
    )
)

df_dias_em_onda.filter("latitude = -23 and longitude = -46").show(10, truncate=False)

In [ ]:
# 1. Identificação dos Eventos de FRIO e Validação da Duração (Global, sem quebra de mês/ano, usando apenas as coordenadas geográfica)
df_eventos_duracao_frio = (
    df_eventos_frio
    .groupBy("latitude", "longitude", "grupo_id")
    .agg(
        F.count("data_medicao").alias("duracao_total_onda"),
        F.min("data_medicao").alias("inicio_onda"),
        F.max("data_medicao").alias("fim_onda")
    )
    .filter(F.col("duracao_total_onda") > 2) # Filtra apenas eventos reais (> 2 dias)
)

# 2. Dias individuais de ondas de frio válidas
df_dias_em_onda_frio = (
    df_eventos_frio
    .join(df_eventos_duracao_frio, ["latitude", "longitude", "grupo_id"], "inner")
    .select(
        "latitude", 
        "longitude", 
        "data_medicao", 
        "ano", 
        "mes", 
        "grupo_id", 
        "duracao_total_onda",
        "valor"  # Mão dupla: temperatura para amplitude e magnitude
    )
)

#### Adicionar as métricas:

- Número de ondas de calor (N-OdC)      : Total de eventos de ondas de calor registrados em um determinado ano.
- Frequência das ondas de calor (F-OdC) : Número total de dias que compõem as ondas de calor ao longo do ano.
- Duração das ondas de calor (D-OdC)    : Duração, em dias, do evento de onda de calor mais longo registrado no ano.
- Amplitude das ondas de calor (A-OdC)  : Maior valor da temperatura média diária observado durante eventos de onda de calor no ano.
- Magnitude das ondas de calor (M-OdC)  : Média da temperatura média diária considerando todos os dias de ocorrência de ondas de calor no ano.

In [ ]:
# Métricas mensais para Ondas de CALOR

df_metricas_mensal = (
    df_dias_em_onda
    .groupBy("latitude", "longitude", "ano", "mes")
    .agg(
        # N-OdC: Número de ondas distintas no mês
        F.countDistinct("grupo_id").alias("numero_ondas_calor"),
        
        # F-OdC: Total exato de dias sob onda de calor DENTRO do mês
        F.count("data_medicao").alias("frequencia_dias_onda_calor"),
        
        # D-OdC: Duração máxima (e média) das ondas no mês
        F.max("duracao_total_onda").alias("duracao_maxima_onda"),
        F.round(F.avg("duracao_total_onda"), 2).alias("duracao_media_ondas"),
        
        # A-OdC: Maior temperatura diária observada em dias de onda de calor no mês
        F.max("valor").alias("amplitude_onda_calor"),
        
        # M-OdC: Média das temperaturas diárias considerando os dias de onda de calor no mês
        F.round(F.avg("valor"), 2).alias("magnitude_onda_calor")
    )
    .orderBy("ano", "mes", "latitude", "longitude")
)

df_metricas_mensal.filter("latitude = -23 and longitude = -46").show(10, truncate=False)

In [ ]:
# Métricas mensais para Ondas de FRIO

df_metricas_mensal_frio = (
    df_dias_em_onda_frio
    .groupBy("latitude", "longitude", "ano", "mes")
    .agg(
        # N-OdF: Número de ondas de frio distintas
        F.countDistinct("grupo_id").alias("numero_ondas_frio"),
        
        # F-OdF: Frequência total de dias em onda de frio no mês
        F.count("data_medicao").alias("frequencia_dias_onda_frio"),
        
        # D-OdF: Duração máxima (e média) das ondas no mês
        F.max("duracao_total_onda").alias("duracao_maxima_onda_frio"),
        F.round(F.avg("duracao_total_onda"), 2).alias("duracao_media_ondas_frio"),
        
        # A-OdF: Menor temperatura diária observada durante as ondas no mês
        F.min("valor").alias("amplitude_onda_frio"),
        
        # M-OdF: Média das temperaturas diárias durante os dias de onda de frio
        F.round(F.avg("valor"), 2).alias("magnitude_onda_frio")
    )
    .orderBy("ano", "mes", "latitude", "longitude")
)

df_metricas_mensal_frio.filter("latitude = -23 and longitude = -46").show(10, truncate=False)


In [ ]:
# Métricas anuais para Ondas de CALOR

df_metricas_anual = (
    df_dias_em_onda
    .groupBy("latitude", "longitude", "ano")
    .agg(
        # N-OdC: Número de ondas distintas no ano
        F.countDistinct("grupo_id").alias("numero_ondas_calor"),
        
        # F-OdC: Total de dias do ano passados sob onda de calor
        F.count("data_medicao").alias("frequencia_dias_onda_calor"),
        
        # D-OdC: Duração máxima do evento no ano
        F.max("duracao_total_onda").alias("duracao_maxima_onda"),
        F.round(F.avg("duracao_total_onda"), 2).alias("duracao_media_ondas"),
        
        # A-OdC: Maior temperatura média diária observada durante ondas no ano
        F.max("valor").alias("amplitude_onda_calor"),
        
        # M-OdC: Média das temperaturas diárias em todos os dias de onda de calor no ano
        F.round(F.avg("valor"), 2).alias("magnitude_onda_calor")
    )
    .orderBy("ano", "latitude", "longitude")
)

df_metricas_anual.filter("latitude = -23 and longitude = -46").show(10, truncate=False)

In [ ]:
# Métricas anuais para Ondas de FRIO

df_metricas_anual_frio = (
    df_dias_em_onda_frio
    .groupBy("latitude", "longitude", "ano")
    .agg(
        # N-OdF: Número de ondas de frio distintas no ano
        F.countDistinct("grupo_id").alias("numero_ondas_frio"),
        
        # F-OdF: Total de dias do ano passados sob onda de frio
        F.count("data_medicao").alias("frequencia_dias_onda_frio"),
        
        # D-OdF: Duração máxima (e média) do evento no ano
        F.max("duracao_total_onda").alias("duracao_maxima_onda_frio"),
        F.round(F.avg("duracao_total_onda"), 2).alias("duracao_media_ondas_frio"),
        
        # A-OdF: Menor temperatura diária registrada durante ondas de frio no ano
        F.min("valor").alias("amplitude_onda_frio"),
        
        # M-OdF: Média das temperaturas nos dias sob onda de frio no ano
        F.round(F.avg("valor"), 2).alias("magnitude_onda_frio")
    )
    .orderBy("ano", "latitude", "longitude")
)

df_metricas_anual_frio.filter("latitude = -23 and longitude = -46").show(10, truncate=False)